# 🥇 Gold CRM & ERP — Star Schema

## Goal
Transform cleaned Silver data into business-ready Gold dimension and fact tables for analytics and reporting.

## Data Model
- gold.dim_customers
- gold.dim_products
- gold.fact_sales
- gold.fact_sales_currency

## Flow
Silver Delta Tables → Dimensions → Fact Tables → Analytics

## 1. Customer Dimension

Combine CRM customer data with ERP customer and location data to create the Gold customer dimension.

In [0]:
from pyspark.sql.functions import col, when, coalesce, lit, row_number
from pyspark.sql.window import Window

ci = spark.table("silver.crm_cust_info")
ca = spark.table("silver.erp_cust_az12")
la = spark.table("silver.erp_loc_a101")

customers_df = ci.alias("ci") \
    .join(
        ca.alias("ca"),
        col("ci.cst_key") == col("ca.CID"),
        "left"
    ) \
    .join(
        la.alias("la"),
        col("ci.cst_key") == col("la.CID"),
        "left"
    )

customer_window = Window.orderBy(col("ci.cst_id"))

customers_df = customers_df.select(
    row_number().over(customer_window).alias("customer_key"),
    col("ci.cst_id").alias("customer_id"),
    col("ci.cst_key").alias("customer_number"),
    col("ci.cst_firstname").alias("first_name"),
    col("ci.cst_lastname").alias("last_name"),
    col("la.CNTRY").alias("country"),
    col("ci.cst_marital_status").alias("marital_status"),

    when(
        col("ci.cst_gndr") != "n/a",
        col("ci.cst_gndr")
    ).otherwise(
        coalesce(col("ca.GEN"), lit("n/a"))
    ).alias("gender"),

    col("ca.BDATE").alias("birthdate"),
    col("ci.cst_create_date").alias("create_date")
)

display(customers_df)

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_customers")

In [0]:
%sql
SELECT *
FROM gold.dim_customers;

## 2. Product Dimension

Combine CRM product data with ERP category data to create the Gold product dimension.

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

pn = spark.table("silver.crm_prd_info")
pc = spark.table("silver.erp_px_cat_g1v2")

products_df = pn.alias("pn") \
    .join(
        pc.alias("pc"),
        col("pn.cat_id") == col("pc.ID"),
        "left"
    ) \
    .filter(col("pn.prd_end_dt").isNull())

product_window = Window.orderBy(
    col("pn.prd_start_dt"),
    col("pn.prd_key")
)

products_df = products_df.select(
    row_number().over(product_window).alias("product_key"),
    col("pn.prd_id").alias("product_id"),
    col("pn.prd_key").alias("product_number"),
    col("pn.prd_nm").alias("product_name"),
    col("pn.cat_id").alias("category_id"),
    col("pc.CAT").alias("category"),
    col("pc.SUBCAT").alias("subcategory"),
    col("pc.MAINTENANCE").alias("maintenance"),
    col("pn.prd_cost").alias("cost"),
    col("pn.prd_line").alias("product_line"),
    col("pn.prd_start_dt").alias("start_date")
)

display(products_df)

In [0]:
products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_products")

In [0]:
%sql
SELECT *
FROM gold.dim_products;

## 3. Sales Fact Table

Create the Gold sales fact table by connecting sales transactions with the customer and product dimensions.

In [0]:
from pyspark.sql.functions import col

sd = spark.table("silver.crm_sales_details")
pr = spark.table("gold.dim_products")
cu = spark.table("gold.dim_customers")

sales_df = sd.alias("sd") \
    .join(
        pr.alias("pr"),
        col("sd.sls_prd_key") == col("pr.product_number"),
        "left"
    ) \
    .join(
        cu.alias("cu"),
        col("sd.sls_cust_id") == col("cu.customer_id"),
        "left"
    )

sales_df = sales_df.select(
    col("sd.sls_ord_num").alias("order_number"),
    col("pr.product_key"),
    col("cu.customer_key"),
    col("sd.sls_order_dt").alias("order_date"),
    col("sd.sls_ship_dt").alias("shipping_date"),
    col("sd.sls_due_dt").alias("due_date"),
    col("sd.sls_sales").alias("sales_amount"),
    col("sd.sls_quantity").alias("quantity"),
    col("sd.sls_price").alias("price")
)

display(sales_df)

In [0]:
sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_sales")

In [0]:
%sql
SELECT *
FROM gold.fact_sales;

## 4. Sales Currency Fact Table

Combine Gold sales data with Silver currency exchange rates to calculate sales amounts in multiple currencies.

In [0]:
from pyspark.sql.functions import col

f = spark.table("gold.fact_sales")
c = spark.table("silver.currency_rates")

sales_currency_df = f.alias("f") \
    .join(
        c.alias("c"),
        (col("f.order_date") == col("c.date")) &
        (col("c.base") == "EUR"),
        "left"
    )

sales_currency_df = sales_currency_df.select(
    col("f.order_number"),
    col("f.product_key"),
    col("f.customer_key"),
    col("f.order_date"),
    col("f.sales_amount").alias("sales_eur"),
    col("c.quote").alias("currency"),
    col("c.rate").alias("exchange_rate"),
    (col("f.sales_amount") * col("c.rate")).alias("converted_sales")
)

display(sales_currency_df)

In [0]:
sales_currency_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_sales_currency")

In [0]:
%sql
SELECT *
FROM gold.fact_sales_currency;